In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from pathlib import Path
import subprocess

In [ ]:
DATA_DIR = '../data/'

In [1]:
with open(os.path.join(GRAPHS_DIR, 'idepix_graph.xml'), 'r') as f:
    idepix_template = f.read()

NameError: name 'os' is not defined

In [ ]:
def run_snap_gpt(graph_path, output_log="gpt_output.log", memory="6G"):
    gpt_executable = str(Path.home() / "esa-snap" / "bin" / "gpt")

    cmd = [gpt_executable, graph_path, "-c", memory]

    try:
        with open(output_log, 'w') as log_file:
            process = subprocess.run(cmd, stdout=log_file, stderr=subprocess.STDOUT, check=True)
        print(f"SNAP processing complete. Output logged to {output_log}")
    except subprocess.CalledProcessError as e:
        print(f"Error: SNAP GPT failed. See log: {output_log}")
        raise

In [ ]:
def create_graph_file(template, output_path="graph.xml", **kwargs):
    content = template.format(**kwargs)
    with open(output_path, "w") as f:
        f.write(content)
    return output_path

In [ ]:
download_dir = os.path.join(DATA_DIR, "s2_downloads")
results_dir = os.path.join(DATA_DIR, "s2_processed")
dir_content = os.listdir(download_dir)

In [ ]:
for fn in dir_content:
    fp = os.path.join(download_dir, fn)  # fn is a tileId

    if fn != '.ipynb_checkpoints' and os.path.isdir(fp):
        # Processing results directory
        results_fp = os.path.join(results_dir, fn)
        os.makedirs(results_fp, exist_ok=True)
        
        safe_dirs = [x for x in os.listdir(fp) if x.endswith('.SAFE')]
        if len(safe_dirs) != 1:  # explicit check, should be only one SAFE file
            raise ValueError(f"Expected exactly one .SAFE file in {fp}, found {len(safe_dirs)}")
        safe_fn = safe_dirs[0]
        safe_fp = os.path.join(fp, safe_fn)
        
        try:
            # Run IdePix process
            print(f"[{fn}] Running Idepix...")
            output_fp = os.path.join(results_fp, f'idepix-{fn}.nc')
            if os.path.exists(output_fp):
                print("Skipping Idepix graph. Result already exists.")
            else:
                graph_path = create_graph_file(idepix_template,
                                               input_file=safe_fp,
                                               output_file=output_fp)
                output_log = os.path.join(results_fp, f'log-idepix-{fn}.log')
                run_snap_gpt(graph_path, output_log)
        except Exception as e:
            print(f"[{fn}] Error processing: {e}")